# Dynamic Withdrawal Strategies - Step-by-Step Test

이 노트북은 Dynamic 전략 구현을 단계별로 테스트합니다.

## Step 1: 라이브러리 및 데이터 로드

In [22]:
import pandas as pd
import numpy as np
import pickle
import warnings
warnings.filterwarnings('ignore')

print("✅ 라이브러리 로드 완료")

✅ 라이브러리 로드 완료


In [23]:
# 벤치마크 데이터 로드
with open('benchmark_data.pkl', 'rb') as f:
    data = pickle.load(f)

print(f"✅ 벤치마크 데이터 로드 완료")
print(f"   데이터 크기: {data.shape}")
print(f"   날짜 범위: {data.index[0].date()} ~ {data.index[-1].date()}")
print(f"\n컬럼:\n{data.columns.tolist()}")

✅ 벤치마크 데이터 로드 완료
   데이터 크기: (6521, 8)
   날짜 범위: 2001-01-03 ~ 2025-12-31

컬럼:
['미국성장주', '국내주식', '미국국채', '미국외국채', '신흥국달러채권', '국내중기채', '국내장기채', '금']


## Step 2: DataPreprocessor 테스트

In [24]:
from withdrawal_backtest import DataPreprocessor, PORTFOLIOS

print(f"DataPreprocessor 임포트 성공")
print(f"\n포트폴리오 목록:")
for i, port_name in enumerate(PORTFOLIOS.keys(), 1):
    port = PORTFOLIOS[port_name]
    print(f"  {i}. {port_name}: 목표 수익률={port['target_return']:.1f}%, 목표 변동성={port['target_risk']:.2f}%")

DataPreprocessor 임포트 성공

포트폴리오 목록:
  1. Port_4.0%: 목표 수익률=4.0%, 목표 변동성=3.75%
  2. Port_5.0%: 목표 수익률=5.0%, 목표 변동성=4.18%
  3. Port_6.0%: 목표 수익률=6.0%, 목표 변동성=5.00%
  4. Port_7.0%: 목표 수익률=7.0%, 목표 변동성=6.05%
  5. Port_8.0%: 목표 수익률=8.0%, 목표 변동성=7.18%
  6. Port_9.0%: 목표 수익률=9.0%, 목표 변동성=8.36%


In [25]:
# DataPreprocessor 실행
try:
    preprocessor = DataPreprocessor(data, add_portfolios=True)
    returns_df, month_starts = preprocessor.get_data()
    
    print(f"✅ DataPreprocessor 완료")
    print(f"   Returns DataFrame: {returns_df.shape}")
    print(f"   Month Starts Series: {month_starts.shape}")
    print(f"\n   Returns 컬럼:")
    print(f"   {returns_df.columns.tolist()[:10]}...")
except Exception as e:
    print(f"❌ 오류: {e}")
    import traceback
    traceback.print_exc()

=== 데이터 전처리 시작 ===
데이터 기간: 2001-01-03 ~ 2025-12-31
총 거래일: 6,521일
벤치마크: 8개

일별 수익률 계산 중... ✅ (6,520개 수익률)

수익률 통계 (연율화):
         연평균수익률   연변동성
미국성장주     12.16  19.90
국내주식      14.40  28.11
미국국채       4.18  11.06
미국외국채      3.34  11.36
신흥국달러채권    7.45  10.59
국내중기채      3.86   2.37
국내장기채      6.16   8.13
금         13.13  19.50

포트폴리오 수익률 계산:
  추가할 포트폴리오: 6개
  ✅ Port_4.0%: 연수익률 4.92%, 연변동성 2.65%
  ✅ Port_5.0%: 연수익률 6.01%, 연변동성 3.81%
  ✅ Port_6.0%: 연수익률 7.10%, 연변동성 5.39%
  ✅ Port_7.0%: 연수익률 8.28%, 연변동성 7.09%
  ✅ Port_8.0%: 연수익률 9.57%, 연변동성 8.95%
  ✅ Port_9.0%: 연수익률 10.87%, 연변동성 10.89%

월초 거래일 식별 중... ✅ (300개월)
첫 10개 월초: [datetime.date(2001, 1, 4), datetime.date(2001, 2, 1), datetime.date(2001, 3, 1), datetime.date(2001, 4, 2), datetime.date(2001, 5, 1), datetime.date(2001, 6, 1), datetime.date(2001, 7, 2), datetime.date(2001, 8, 1), datetime.date(2001, 9, 3), datetime.date(2001, 10, 1)]
✅ 전처리 완료

✅ DataPreprocessor 완료
   Returns DataFrame: (6520, 14)
   Month Starts Series: (6520,)

   Ret

## Step 3: Dynamic Simulator 테스트

In [26]:
from dynamic_simulator import GuardrailsWithdrawal, GuytonKlingerWithdrawal, DynamicWithdrawalSimulator

print("✅ Dynamic Simulator 클래스 임포트 성공")

# DynamicWithdrawalSimulator 생성
simulator = DynamicWithdrawalSimulator(returns_df, month_starts)
print(f"✅ DynamicWithdrawalSimulator 생성 완료")
print(f"   총 날짜: {len(simulator.dates)}")
print(f"   월초 개수: {np.sum(month_starts)}")

✅ Dynamic Simulator 클래스 임포트 성공
✅ DynamicWithdrawalSimulator 생성 완료
   총 날짜: 6520
   월초 개수: 300


## Step 4: Guardrails 백테스트 (간단한 테스트)

In [27]:
# 하나의 포트폴리오와 인출률로 빠른 테스트
test_portfolio = 'Port_5.0%'
test_wr = 0.05  # 5%
horizon_years = 10

print(f"테스트 설정:")
print(f"  포트폴리오: {test_portfolio}")
print(f"  인출률: {test_wr*100:.1f}%")
print(f"  기간: {horizon_years}년")

try:
    results_df = simulator.run_guardrails_backtest(
        benchmark=test_portfolio,
        horizon_years=horizon_years,
        initial_wr=test_wr,
        guardrail_width=0.20,
        inflation_rate=0.02,
        v0=100.0,
        verbose=True
    )
    
    print(f"\n✅ Guardrails 백테스트 완료")
    print(f"\n결과 DataFrame:")
    print(f"  크기: {results_df.shape}")
    print(f"  컬럼: {results_df.columns.tolist()}")
    print(f"\n샘플 데이터 (첫 5행):")
    display(results_df[['start_date', 'terminal_nav', 'total_withdrawal', 'is_fail']].head())
    
except Exception as e:
    print(f"❌ 오류: {e}")
    import traceback
    traceback.print_exc()

테스트 설정:
  포트폴리오: Port_5.0%
  인출률: 5.0%
  기간: 10년

Guardrails 백테스트: Port_5.0%, 10년, WR=5.0%
총 경로 수: 3,911


완료: 3,911개 경로

✅ Guardrails 백테스트 완료

결과 DataFrame:
  크기: (3911, 6)
  컬럼: ['start_date', 'terminal_nav', 'total_withdrawal', 'withdrawal_path', 'nav_path', 'is_fail']

샘플 데이터 (첫 5행):


,start_date,terminal_nav,total_withdrawal,is_fail
0,2001-01-04,115.287744,58.086801,False
1,2001-01-05,112.404324,56.163778,False
2,2001-01-08,112.404324,56.163778,False
3,2001-01-09,112.404324,56.163778,False
4,2001-01-10,112.404324,56.163778,False


## Step 5: 메트릭 계산 테스트

In [28]:
from metrics_calculator import MetricsCalculator

metrics_calc = MetricsCalculator()

try:
    metrics = metrics_calc.calculate_optimization_metrics(results_df, v0=100.0)
    
    print(f"✅ 메트릭 계산 완료")
    print(f"\n주요 메트릭:")
    print(f"  총 인출액 (평균): {metrics['total_withdrawal_mean']:,.2f}")
    print(f"  총 인출액 (범위): {metrics['total_withdrawal_worst']:,.2f} ~ {metrics['total_withdrawal_best']:,.2f}")
    print(f"  YoY 변동성 (평균): {metrics['yoy_volatility_mean']:.4f}")
    print(f"  YoY 변동성 (90분위): {metrics['yoy_volatility_90pct']:.4f}")
    print(f"  실패율: {metrics['failure_rate']:.2%}")
    print(f"  최종 NAV (중앙값): {metrics['terminal_nav_median']:.2%}")
    print(f"  경로 수: {metrics['total_paths']}")
    
except Exception as e:
    print(f"❌ 오류: {e}")
    import traceback
    traceback.print_exc()

✅ 메트릭 계산 완료

주요 메트릭:
  총 인출액 (평균): 57.16
  총 인출액 (범위): 52.48 ~ 61.51
  YoY 변동성 (평균): 0.0260
  YoY 변동성 (90분위): 0.0386
  실패율: 28.97%
  최종 NAV (중앙값): 10436.97%
  경로 수: 3911


## Step 6: Guyton-Klinger 백테스트

In [29]:
try:
    gk_results_df = simulator.run_guyton_klinger_backtest(
        benchmark=test_portfolio,
        horizon_years=horizon_years,
        initial_wr=test_wr,
        guardrail_width=0.20,
        adjustment_pct=0.10,
        freeze_threshold=-0.10,
        inflation_rate=0.02,
        v0=100.0,
        verbose=True
    )
    
    print(f"\n✅ Guyton-Klinger 백테스트 완료")
    print(f"\n결과 DataFrame:")
    print(f"  크기: {gk_results_df.shape}")
    print(f"\n샘플 데이터 (첫 5행):")
    display(gk_results_df[['start_date', 'terminal_nav', 'total_withdrawal', 'is_fail']].head())
    
except Exception as e:
    print(f"❌ 오류: {e}")
    import traceback
    traceback.print_exc()


Guyton-Klinger 백테스트: Port_5.0%, 10년, WR=5.0%
총 경로 수: 3,911


완료: 3,911개 경로

✅ Guyton-Klinger 백테스트 완료

결과 DataFrame:
  크기: (3911, 6)

샘플 데이터 (첫 5행):


,start_date,terminal_nav,total_withdrawal,is_fail
0,2001-01-04,115.287744,58.086801,False
1,2001-01-05,112.404324,56.163778,False
2,2001-01-08,112.404324,56.163778,False
3,2001-01-09,112.404324,56.163778,False
4,2001-01-10,112.404324,56.163778,False


## Step 6.1: Guyton-Klinger 일별 Path 값 조회

`gk_results_df`의 `withdrawal_path`/`nav_path`는 월별 배열이므로, `get_single_path_detail`을 사용하여 일별 경로 데이터를 조회합니다.

In [38]:
# ============================================================
# 시작일 설정 - 원하는 날짜로 변경하세요
# ============================================================
start_date = '2008-01-02'  # 원하는 시작일로 변경

# 자산 표시명 → 원본 data 컬럼명 매핑 (가격 지수용)
PRICE_COL_MAP = {
    'Korean_Equity':          '국내주식',
    'US_Growth':              '미국성장주',
    'Korean_Bond_Composite':  '국내중기채',
    'Korean_Bond_10Y':        '국내장기채',
    'EM_Dollar_Bond':         '신흥국달러채권',
    'US_Bond':                '미국국채',
    'Global_ex_US_Bond':      '미국외국채',
    'Gold':                   '금',
}

# 포트폴리오 가중치 매핑 (한글 → 영문)
WEIGHT_MAP = {
    '한국주식': 'Korean_Equity',
    '미국성장주': 'US_Growth',
    '한국종합채권': 'Korean_Bond_Composite',
    '한국국고채10년': 'Korean_Bond_10Y',
    '신흥국달러채권': 'EM_Dollar_Bond',
    '미국채권': 'US_Bond',
    '미국외글로벌채권': 'Global_ex_US_Bond',
    '금': 'Gold',
}

# ============================================================
# 포트폴리오 상세 정보 출력
# ============================================================
print(f"\n{'='*60}")
print(f"포트폴리오 상세 정보")
print(f"{'='*60}")

portfolio_config = PORTFOLIOS.get(test_portfolio)
if portfolio_config:
    print(f"\n포트폴리오: {test_portfolio}")
    print(f"목표 수익률: {portfolio_config['target_return']:.2f}%")
    print(f"목표 변동성: {portfolio_config['target_risk']:.2f}%")
    print(f"\n자산 구성:")
    
    total_weight = 0.0
    for asset_kor, weight_pct in portfolio_config['weights'].items():
        asset_eng = WEIGHT_MAP.get(asset_kor, asset_kor)
        print(f"  {asset_kor:15s} ({asset_eng:25s}): {weight_pct:6.2f}%")
        total_weight += weight_pct
    
    print(f"  {'-'*50}")
    print(f"  {'합계':15s} {' ':26s}: {total_weight:6.2f}%")
else:
    print(f"⚠️  포트폴리오 '{test_portfolio}' 설정을 찾을 수 없습니다.")

# ============================================================
# 일별 경로 조회 (get_single_path_detail)
# ============================================================
print(f"\n{'='*60}")
print(f"일별 경로 데이터 조회 (start_date: {start_date})")
print(f"{'='*60}")

daily_path_df = simulator.get_single_path_detail(
    portfolio=test_portfolio,
    start_date=start_date,
    strategy='guyton_klinger',
    horizon_years=horizon_years,
    initial_wr=test_wr,
    guardrail_width=0.20,
    adjustment_pct=0.10,
    freeze_threshold=-0.10,
    inflation_rate=0.02,
    v0=100.0
)

# ============================================================
# 추가 1: 자산별 Price Index Level 추가
# ============================================================
dates_in_path = daily_path_df['Date'].values
for display_name, data_col in PRICE_COL_MAP.items():
    col_name = f'Price_{display_name}'
    daily_path_df[col_name] = data.loc[dates_in_path, data_col].values

# ============================================================
# 추가 2: 포트폴리오 자산별 비중 추가
# ============================================================
portfolio_config = PORTFOLIOS.get(test_portfolio)
if portfolio_config:
    portfolio_weights = portfolio_config['weights']  # % 단위
    
    # 8개 자산 모두 0으로 초기화
    for display_name in PRICE_COL_MAP.keys():
        daily_path_df[f'Weight_{display_name}'] = 0.0
    
    # 실제 비중 적용
    for asset_kor, weight_pct in portfolio_weights.items():
        if asset_kor in WEIGHT_MAP:
            display_name = WEIGHT_MAP[asset_kor]
            daily_path_df[f'Weight_{display_name}'] = weight_pct  # % 단위

# ============================================================
# 일별 데이터 출력
# ============================================================
print(f"\n일별 경로 DataFrame:")
print(f"  크기: {daily_path_df.shape}")
print(f"  날짜 범위: {daily_path_df['Date'].iloc[0].date()} ~ {daily_path_df['Date'].iloc[-1].date()}")
print(f"  총 거래일: {len(daily_path_df)}")

# NAV, Price, Weight 컬럼 확인
nav_cols = [c for c in daily_path_df.columns if c.startswith('NAV_')]
price_cols = [c for c in daily_path_df.columns if c.startswith('Price_')]
weight_cols = [c for c in daily_path_df.columns if c.startswith('Weight_')]

print(f"\n컬럼 구성:")
print(f"  NAV 컬럼: {len(nav_cols)}개")
print(f"  Price 컬럼: {len(price_cols)}개")
print(f"  Weight 컬럼: {len(weight_cols)}개")

# 주요 컬럼 표시
display_cols = ['Date', 'Total_NAV'] + nav_cols + price_cols + weight_cols + [
    'Monthly_Withdrawal', 'Is_Month_Start', 'Guardrail_Status'
]

print(f"\n--- 일별 데이터 (NAV, Price Index, Weight) ---")
display(daily_path_df[display_cols])


포트폴리오 상세 정보

포트폴리오: Port_5.0%
목표 수익률: 5.00%
목표 변동성: 4.18%

자산 구성:
  한국주식            (Korean_Equity            ):   3.55%
  미국성장주           (US_Growth                ):  12.83%
  한국종합채권          (Korean_Bond_Composite    ):  75.87%
  신흥국달러채권         (EM_Dollar_Bond           ):   0.15%
  금               (Gold                     ):   7.60%
  --------------------------------------------------
  합계                                        : 100.00%

일별 경로 데이터 조회 (start_date: 2008-01-02)

일별 경로 DataFrame:
  크기: (2610, 37)
  날짜 범위: 2008-01-02 ~ 2018-01-02
  총 거래일: 2610

컬럼 구성:
  NAV 컬럼: 8개
  Price 컬럼: 8개
  Weight 컬럼: 8개

--- 일별 데이터 (NAV, Price Index, Weight) ---


,Date,Total_NAV,NAV_Korean_Equity,NAV_US_Growth,NAV_Korean_Bond_Composite,NAV_Korean_Bond_10Y,NAV_EM_Dollar_Bond,NAV_US_Bond,NAV_Global_ex_US_Bond,NAV_Gold,...,Weight_US_Growth,Weight_Korean_Bond_Composite,Weight_Korean_Bond_10Y,Weight_EM_Dollar_Bond,Weight_US_Bond,Weight_Global_ex_US_Bond,Weight_Gold,Monthly_Withdrawal,Is_Month_Start,Guardrail_Status
0,2008-01-02,100.000000,3.550000,12.830000,75.870000,0.0,0.150000,0.0,0.0,7.600000,...,12.83,75.87,0.0,0.15,0.0,0.0,7.6,0.416667,False,Normal
1,2008-01-03,99.820990,3.464736,12.645492,75.883030,0.0,0.150308,0.0,0.0,7.677426,...,12.83,75.87,0.0,0.15,0.0,0.0,7.6,0.416667,False,Normal
2,2008-01-04,100.104089,3.461972,12.669308,75.909662,0.0,0.150649,0.0,0.0,7.912498,...,12.83,75.87,0.0,0.15,0.0,0.0,7.6,0.416667,False,Normal
3,2008-01-07,99.708549,3.469910,12.309157,75.925141,0.0,0.150985,0.0,0.0,7.853357,...,12.83,75.87,0.0,0.15,0.0,0.0,7.6,0.416667,False,Normal
4,2008-01-08,99.485562,3.393701,12.283344,75.808605,0.0,0.150995,0.0,0.0,7.848917,...,12.83,75.87,0.0,0.15,0.0,0.0,7.6,0.416667,False,Normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2605,2017-12-27,102.536666,2.935930,21.764941,69.880512,0.0,0.196540,0.0,0.0,7.758743,...,12.83,75.87,0.0,0.15,0.0,0.0,7.6,0.519596,False,Upper_Breach
2606,2017-12-28,102.646543,2.999423,21.717869,69.953437,0.0,0.196097,0.0,0.0,7.779717,...,12.83,75.87,0.0,0.15,0.0,0.0,7.6,0.519596,False,Upper_Breach
2607,2017-12-29,102.718668,3.058340,21.702498,69.970147,0.0,0.195434,0.0,0.0,7.792249,...,12.83,75.87,0.0,0.15,0.0,0.0,7.6,0.519596,False,Upper_Breach
2608,2018-01-01,102.083101,3.042852,21.451937,69.615807,0.0,0.194379,0.0,0.0,7.778126,...,12.83,75.87,0.0,0.15,0.0,0.0,7.6,0.519596,True,Upper_Breach


In [39]:
daily_path_df.to_excel('guyton_klinger_path_details.xlsx', index=False)

## Step 6.5: Fixed Rate 백테스트

In [ ]:
try:
    fixed_results_df = simulator.run_fixed_backtest(
        benchmark=test_portfolio,
        horizon_years=horizon_years,
        initial_wr=test_wr,
        inflation_rate=0.02,
        v0=100.0,
        verbose=True
    )
    
    print(f"\n✅ Fixed Rate 백테스트 완료")
    print(f"\n결과 DataFrame:")
    print(f"  크기: {fixed_results_df.shape}")
    print(f"\n샘플 데이터 (첫 5행):")
    display(fixed_results_df[['start_date', 'terminal_nav', 'total_withdrawal', 'is_fail']].head())
    
except Exception as e:
    print(f"❌ 오류: {e}")
    import traceback
    traceback.print_exc()

## Step 7: WithdrawalOptimizer 테스트

In [ ]:
from optimizer import WithdrawalOptimizer

optimizer = WithdrawalOptimizer(returns_df, month_starts)

print(f"✅ WithdrawalOptimizer 생성 완료")

# 작은 범위로 빠른 테스트
withdrawal_rates = np.array([0.04, 0.05, 0.06])
constraints = {
    'max_yoy_volatility': 0.20,
    'max_failure_rate': 0.10,
    'min_terminal_nav': 0.40
}

print(f"\n테스트 설정:")
print(f"  인출률: {withdrawal_rates}")
print(f"  제약조건:")
print(f"    - Max YoY Volatility: {constraints['max_yoy_volatility']:.0%}")
print(f"    - Max Failure Rate: {constraints['max_failure_rate']:.0%}")
print(f"    - Min Terminal NAV: {constraints['min_terminal_nav']:.0%}")

In [ ]:
try:
    # 단일 포트폴리오 최적화 (빠른 테스트)
    portfolio_results = optimizer.optimize_single_portfolio(
        portfolio_name='Port_5.0%',
        withdrawal_rates=withdrawal_rates,
        horizon_years=10,
        constraints=constraints,
        v0=100.0,
        verbose=True
    )
    
    print(f"\n✅ 단일 포트폴리오 최적화 완료")
    print(f"\n결과 구조:")
    for wr, strategies in portfolio_results.items():
        print(f"  인출률 {wr:.2%}:")
        for strategy, metrics in strategies.items():
            feasible = "✓" if metrics.get('is_feasible') else "✗"
            total_w = metrics.get('total_withdrawal_mean', 0)
            print(f"    {strategy:20s} {feasible} - 총 인출액: {total_w:,.2f}")
    
except Exception as e:
    print(f"❌ 오류: {e}")
    import traceback
    traceback.print_exc()

## Step 8: 전체 최적화 (2개 포트폴리오)

In [ ]:
try:
    # 작은 범위로 빠른 테스트
    test_portfolios = ['Port_4.0%', 'Port_5.0%']
    test_wrs = np.array([0.04, 0.05, 0.06])
    
    results_df = optimizer.optimize_all_portfolios(
        portfolio_names=test_portfolios,
        withdrawal_rates=test_wrs,
        horizon_years=10,
        constraints=constraints,
        v0=100.0
    )
    
    print(f"\n✅ 전체 최적화 완료")
    print(f"\n결과 DataFrame: {results_df.shape}")
    print(f"\n전체 결과:")
    display(results_df.head(10))
    
except Exception as e:
    print(f"❌ 오류: {e}")
    import traceback
    traceback.print_exc()

## Step 9: 결과 요약

In [ ]:
try:
    print(f"\n최적화 결과 통계:")
    print(f"  전체 시나리오: {len(results_df)}")
    print(f"  가능한 시나리오 (제약 만족): {len(results_df[results_df['Feasible']==True])}")
    print(f"  불가능한 시나리오: {len(results_df[results_df['Feasible']==False])}")
    
    print(f"\n포트폴리오별 최적 솔루션:")
    for portfolio in results_df['Portfolio'].unique():
        df_port = results_df[results_df['Portfolio'] == portfolio]
        for strategy in ['fixed', 'guardrails', 'guyton_klinger']:
            df_strat = df_port[df_port['Strategy'] == strategy]
            df_feasible = df_strat[df_strat['Feasible'] == True]
            if not df_feasible.empty:
                best = df_feasible.loc[df_feasible['Total_Withdrawal'].idxmax()]
                print(f"  {portfolio:12s} {strategy:20s}: WR={best['WR']:.2%}, "
                      f"TotalW={best['Total_Withdrawal']:.0f}, "
                      f"Vol={best['YoY_Volatility']:.2%}")
            else:
                print(f"  {portfolio:12s} {strategy:20s}: No feasible solution")
                
except Exception as e:
    print(f"❌ 오류: {e}")
    import traceback
    traceback.print_exc()

## Step 10: UI 컴포넌트 테스트

In [ ]:
try:
    from dynamic_strategy_ui import (
        create_pareto_frontier_chart,
        create_strategy_comparison_chart
    )
    
    print(f"✅ UI 컴포넌트 임포트 성공")
    
    # Pareto Frontier 차트 생성
    portfolio_chart = 'Port_5.0%'
    fig = create_pareto_frontier_chart(results_df, portfolio_chart)
    
    print(f"✅ Pareto Frontier 차트 생성 완료")
    print(f"   차트 타입: {type(fig)}")
    
except Exception as e:
    print(f"❌ 오류: {e}")
    import traceback
    traceback.print_exc()

## Step 11: 특정 경로 상세 조회 (get_single_path_detail)

단일 시작일의 일별 경로 데이터를 반환하는 기능 테스트 (3가지 전략, 8개 자산 NAV 추적)

In [ ]:
# This cell is no longer needed - functionality merged into Step 11

In [ ]:
from dynamic_simulator import validate_backtest_consistency

print("\n" + "="*60)
print("백테스트 vs single_path_detail 일관성 검증")
print("="*60)

# 검증할 전략 및 백테스트 결과
strategies_to_validate = [
    ('fixed', fixed_results_df),
    ('guardrails', results_df),
    ('guyton_klinger', gk_results_df)
]

for strategy_name, backtest_df in strategies_to_validate:
    print(f"\n{'='*60}")
    print(f"{strategy_name.upper()} 전략 검증")
    print(f"{'='*60}")
    
    try:
        if strategy_name == 'fixed':
            result = validate_backtest_consistency(
                simulator=simulator,
                backtest_results=backtest_df,
                portfolio=test_portfolio,
                strategy='fixed',
                horizon_years=horizon_years,
                initial_wr=test_wr,
                inflation_rate=0.02,
                v0=100.0,
                sample_size=3,
                tolerance=1e-4
            )
        elif strategy_name == 'guardrails':
            result = validate_backtest_consistency(
                simulator=simulator,
                backtest_results=backtest_df,
                portfolio=test_portfolio,
                strategy='guardrails',
                horizon_years=horizon_years,
                initial_wr=test_wr,
                guardrail_width=0.20,
                inflation_rate=0.02,
                v0=100.0,
                sample_size=3,
                tolerance=1e-4
            )
        else:  # guyton_klinger
            result = validate_backtest_consistency(
                simulator=simulator,
                backtest_results=backtest_df,
                portfolio=test_portfolio,
                strategy='guyton_klinger',
                horizon_years=horizon_years,
                initial_wr=test_wr,
                guardrail_width=0.20,
                adjustment_pct=0.10,
                freeze_threshold=-0.10,
                inflation_rate=0.02,
                v0=100.0,
                sample_size=3,
                tolerance=1e-4
            )
        
        if result['all_passed']:
            print(f"✅ 모든 검증 통과: {result['n_passed']}/{result['n_tested']}")
        else:
            print(f"❌ 일부 검증 실패: {result['n_passed']}/{result['n_tested']}")
            print(f"\n실패한 케이스:")
            for case in result['failed_cases']:
                print(f"  - 경로 {case['idx']}: {case['error']}")
                
    except Exception as e:
        print(f"❌ 검증 중 오류 발생: {e}")
        import traceback
        traceback.print_exc()

print(f"\n{'='*60}")
print(f"전체 일관성 검증 완료")
print(f"{'='*60}")

In [ ]:
# Guyton-Klinger 테스트
try:
    path_df_gk = simulator.get_single_path_detail(
        portfolio=test_portfolio,
        start_date=test_start_date,
        strategy='guyton_klinger',
        horizon_years=10,
        initial_wr=0.05,
        guardrail_width=0.20,
        adjustment_pct=0.10,
        freeze_threshold=-0.10,
        inflation_rate=0.02,
        v0=100.0
    )
    
    print(f"\u2705 Guyton-Klinger 일별 경로 데이터 생성 완료")
    print(f"  크기: {path_df_gk.shape}")
    print(f"  날짜 범위: {path_df_gk['Date'].iloc[0].date()} ~ {path_df_gk['Date'].iloc[-1].date()}")
    
    # 검증
    path_df_gk['Calculated_Total'] = (
        path_df_gk['NAV_Korean_Equity'] + 
        path_df_gk['NAV_US_Growth'] + 
        path_df_gk['NAV_Bond'] + 
        path_df_gk['NAV_Gold']
    )
    max_diff_gk = abs(path_df_gk['Total_NAV'] - path_df_gk['Calculated_Total']).max()
    print(f"  Total_NAV 최대 오차: {max_diff_gk:.10f}")
    
    print(f"  Guardrail 상태 분포:")
    print(path_df_gk['Guardrail_Status'].value_counts())
    
    # 엑셀 저장
    output_path_gk = f'path_{test_start_date}_guyton_klinger.xlsx'
    path_df_gk.drop(columns=['Calculated_Total'], inplace=True)
    path_df_gk.to_excel(output_path_gk, index=False)
    print(f"\u2705 엑셀 저장 완료: {output_path_gk}")
    
except Exception as e:
    print(f"\u274c 오류: {e}")
    import traceback
    traceback.print_exc()